# train.ipynb — Seq2Seq Num2Text (1 enc 3 dec) 학습 · 평가 · 저장

이 노트북 **하나만** 있으면 Colab 에서 처음부터 끝까지 실행된다. (`src/` 를 import 하거나 저장소를 clone 하지 않는다)

- `src/` 의 `data / model / seq2seq / inference / checkpoint / train` 코드를 셀로 옮겨 놓았다. 로컬 실행은 `python src/main.py`.
- 데이터(`num2kor.txt`, `num2eng.txt`)는 노트북 옆에 `data/` 가 없으면 GitHub 에서 자동으로 내려받는다.

**실행**: 런타임 유형을 T4 GPU 로 바꾸고 위에서부터 순서대로 실행한다.

| 단계 | 내용 |
|---|---|
| 1 ~ 3 | 환경, 설정, 데이터 파일 준비 |
| 4 ~ 9 | 코드 정의 (data, model, seq2seq, inference, checkpoint, train) |
| 10 ~ 12 | loader · 모델 생성, 학습 (실행할 때마다 저장된 지점부터 이어서 학습) |
| 13 ~ 15 | 방향별 test 평가, 제출용 `.pt` 6개 저장, 저장 파일로 번역 확인 · 직접 입력 |

## 1. 환경 준비

In [ ]:
# Colab / 로컬 둘 다 동작. GPU 가 없으면 CPU 로 실행한다.
import functools
import json
import os
import random
import sys
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset, Subset

IN_COLAB = "google.colab" in sys.modules
WORK_DIR = "/content" if IN_COLAB else os.getcwd()

# 학습 상태 저장 위치: Colab 은 Drive (세션이 끊겨도 남는다), 연결이 안 되면 세션 안
CKPT_DIR = os.path.join(WORK_DIR, "checkpoints")
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        CKPT_DIR = "/content/drive/MyDrive/2026-nlp/checkpoints"
    except Exception as e:
        print("[안내] Drive 연결 실패 -> 이 세션 안에만 저장합니다:", e)
SUBMIT_DIR = os.path.join(CKPT_DIR, "final")   # 제출용 .pt 6개
os.makedirs(CKPT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device)
if device == "cpu":
    print("[안내] GPU 가 없습니다. Colab 이면 런타임 > 런타임 유형 변경 > T4 GPU 로 바꾸세요.")
    print("       CPU 로 전체 학습은 매우 오래 걸립니다. 동작 확인만 하려면 다음 셀의 LIMIT 을 96 정도로 두세요.")
print("저장 위치:", CKPT_DIR)

## 2. 설정

`src/config.json` 과 같은 값이다. 단, epoch 수는 12번 셀의 `EPOCHS_PER_RUN` 으로 정한다.

In [ ]:
CONFIG = {
    "train_ratio": 0.8,
    "valid_ratio": 0.1,
    "test_ratio": 0.1,
    "random_seed": 42,
    "batch_size": 32,
    "lr": 0.001,
    "grad_clip": 1.0,
    "max_len": 60,
    "model": {"embedding_dim": 128, "hidden_dim": 256, "num_layers": 2, "dropout": 0.1},
}
MODEL_KWARGS = dict(CONFIG["model"])   # Encoder / Decoder 공통 인자

LIMIT = None    # 숫자를 넣으면 분할마다 앞에서 그 개수만 사용 (빠른 동작 확인용, 예: 96)
FRESH = False   # True 면 12번 셀이 저장된 last.pt, last.best.pt 를 지우고 처음부터 학습 (한 번 실행한 뒤 False 로 되돌릴 것)

SEED = CONFIG["random_seed"]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 3. 데이터 파일 준비

In [ ]:
# 노트북 옆(data/, ../data/)에 있으면 그대로 쓰고, 없으면 GitHub 에서 txt 두 개만 내려받는다.
DATA_URL = "https://raw.githubusercontent.com/sloth2003/2026-nlp/main/project01-dev/data/"
DATA_FILES = ("num2kor.txt", "num2eng.txt")


def prepare_data_dir():
    for cand in ("data", os.path.join("..", "data")):
        if all(os.path.isfile(os.path.join(cand, name)) for name in DATA_FILES):
            return Path(cand).resolve()

    target = Path(WORK_DIR) / "data"
    target.mkdir(parents=True, exist_ok=True)
    for name in DATA_FILES:
        path = target / name
        if not path.is_file():
            print("내려받는 중:", DATA_URL + name)
            tmp = path.with_name(name + ".tmp")
            try:
                urllib.request.urlretrieve(DATA_URL + name, tmp)
            except Exception as e:
                raise RuntimeError(
                    f"{name} 을 내려받지 못했습니다 ({e}). "
                    f"파일을 직접 {target} 폴더에 올린 뒤 이 셀을 다시 실행하세요."
                )
            tmp.replace(path)   # 다 받은 뒤에만 제 이름으로 바꾼다
    return target


DATA_DIR = prepare_data_dir()
print("데이터 폴더:", DATA_DIR)

## 4. data — 데이터 · Vocab · 6방향 loader (담당: 재후)

In [ ]:
# ===== src/data.py =====
# set seed
def seed_everything(seed=42):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)  # CPU + GPU 모두 고정

# data
num_kor = pd.read_csv(
    DATA_DIR / 'num2kor.txt',
    sep='\t',
    header=None,
    names=['num', 'kor']
)

num_eng = pd.read_csv(
    DATA_DIR / 'num2eng.txt',
    sep='\t',
    header=None,
    names=['num', 'eng']
)

# combined df
# num2kor / num2eng 는 서로 다른 숫자를 랜덤 샘플링한 것이라 숫자가 일부만 겹친다.
# outer merge 라서 한쪽에만 있는 숫자는 kor 또는 eng 가 NaN -> build_loaders 에서 쓰는 두 열 기준으로 걸러낸다.
combined = pd.merge(num_kor, num_eng, on='num', how='outer')
combined['num'] = combined['num'].astype('str')


# Vocab token <-> word
PAD, BOS, EOS, UNK = '<pad>', '<bos>', '<eos>', '<unk>'
SPECIALS = [PAD, BOS, EOS, UNK]
PAD_ID, BOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3

def tokenize(text, col): 
    # 영어: 단어 단위 / 숫자·한국어: 글자 단위
    return text.split() if col == 'eng' else list(text)

class Vocab:
    def __init__(self, sentences, col):
        self.col = col
        tokens = sorted({t for s in sentences for t in tokenize(s, col)})
        self.itos = SPECIALS + tokens                        # id → 토큰 (index를 id로 삼음)
        self.stoi = {t: i for i, t in enumerate(self.itos)}  # 토큰 → id

    def __len__(self):
        return len(self.itos)

    def encode(self, text):
        ids = [self.stoi.get(t, UNK_ID) for t in tokenize(text, self.col)]  # 모르는 토큰 → <unk>
        return [BOS_ID] + ids + [EOS_ID]

    def decode(self, ids):
        toks = []
        for i in ids:
            if i == EOS_ID:              # <eos> 이후는 버림
                break
            if i in (PAD_ID, BOS_ID):
                continue
            toks.append(self.itos[i])    # <unk>는 그대로 보이게 둠
        return ' '.join(toks) if self.col == 'eng' else ''.join(toks)


# type for indexing data
class NumDataset(Dataset):
    def __init__(self, df, input_col, output_col, src_vocab, tgt_vocab):
        # df        : DataFrame
        # input_col : 입력 열 이름 (예: 'num')
        # output_col: 정답 열 이름 (예: 'kor')
        # src_vocab : 입력 언어 사전 / tgt_vocab: 출력 언어 사전
        self.src = [torch.tensor(src_vocab.encode(s)) for s in df[input_col]]
        self.tgt = [torch.tensor(tgt_vocab.encode(s)) for s in df[output_col]]

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return self.src[idx], self.tgt[idx]


def collate_fn(batch):
    src, tgt = zip(*batch)
    src = pad_sequence(src, batch_first=True, padding_value=PAD_ID)
    tgt = pad_sequence(tgt, batch_first=True, padding_value=PAD_ID)
    return src, tgt


def pair_df(df, input_col, output_col):
    # 입력·정답 열이 둘 다 있는 행만 남김 (한쪽 파일에만 있는 숫자 제외)
    return df.dropna(subset=[input_col, output_col]).reset_index(drop=True)


# 1 enc 3 dec 학습용 loader 묶음
def split_by_number(nums, config):
    """숫자 -> 'train' / 'valid' / 'test'. 숫자 기준으로 한 번만 나눠 6방향이 같은 분할을 쓴다."""
    nums = sorted(nums)   # 문자열 그대로 정렬 (int 변환 없음)
    g = torch.Generator().manual_seed(config['random_seed'])
    order = torch.randperm(len(nums), generator=g).tolist()
    n_train = int(len(nums) * config['train_ratio'])
    n_valid = int(len(nums) * config['valid_ratio'])
    split_of = {}
    for rank, i in enumerate(order):
        split_of[nums[i]] = 'train' if rank < n_train else 'valid' if rank < n_train + n_valid else 'test'
    return split_of


def build_vocabs(df, langs=('num', 'eng', 'kor')):
    """출력 vocab 은 언어별, 입력 vocab 은 세 언어 토큰을 합친 통합 사전.

    src_vocabs 의 세 Vocab 은 id 가 모두 같고 col(토큰화 방식)만 다르다 -> 공유 Encoder 하나에 넣을 수 있다.
    """
    tgt_vocabs = {lang: Vocab(df[lang].dropna(), lang) for lang in langs}
    joint_tokens = sorted({t for v in tgt_vocabs.values() for t in v.itos[len(SPECIALS):]})

    src_vocabs = {}
    for lang in langs:
        v = Vocab([], lang)
        v.itos = SPECIALS + joint_tokens
        v.stoi = {t: i for i, t in enumerate(v.itos)}
        src_vocabs[lang] = v
    return src_vocabs, tgt_vocabs


def build_loaders(config, directions, langs=('num', 'eng', 'kor'), df=None, limit=None):
    """6방향 loader 묶음 -> loaders, src_vocabs, tgt_vocabs

    loaders[direction]['train' / 'valid' / 'test'] 로 꺼낸다 (direction: 'kor2eng' 등).
    limit: 숫자를 주면 분할마다 앞에서 그 개수만 사용 (빠른 동작 확인용).
    """
    df = combined if df is None else df
    split_of = split_by_number(df['num'], config)
    src_vocabs, tgt_vocabs = build_vocabs(df, langs)

    loaders = {}
    for direction in directions:
        s, t = direction.split('2')
        pair = pair_df(df, s, t)   # 두 열이 다 있는 행만 (kor-eng 는 공통 숫자만)
        dataset = NumDataset(pair, s, t, src_vocabs[s], tgt_vocabs[t])
        loaders[direction] = {}
        for part in ('train', 'valid', 'test'):
            idx = [i for i, n in enumerate(pair['num']) if split_of[n] == part]
            if limit is not None:
                idx = idx[:limit]
            loaders[direction][part] = DataLoader(
                Subset(dataset, idx), batch_size=config['batch_size'],
                shuffle=(part == 'train'), collate_fn=collate_fn,
            )
    return loaders, src_vocabs, tgt_vocabs


print('num2kor', num_kor.shape, '| num2eng', num_eng.shape, '| combined', combined.shape)

## 5. model — GRU Encoder / Decoder (담당: 기수)

```
구조 (attention 미사용)
  Encoder: Embedding -> Packing -> 양방향 GRU -> Unpacking -> 방향 합산
  Decoder: Embedding -> Packing -> GRU -> Unpacking -> FC

- seq2seq.py 에서 공유 Encoder 1개와 출력 언어별 Decoder 3개를 생성한다.
- Encoder 의 정방향 / 역방향 출력과 hidden 을 합산해 기존 인터페이스를 유지한다.
- 합산한 hidden 을 단방향 Decoder 의 초기 hidden 으로 전달한다.
- Decoder 는 softmax 전 logits 를 반환한다 (Cross-Entropy Loss 입력).

기본 설정: embedding_dim=128, hidden_dim=256, num_layers=2, dropout=0.1
입력은 (batch, 길이)의 토큰 ID 이며, PAD 는 오른쪽에 붙인다.
```

In [ ]:
# ===== src/model.py =====
# ---------------------------------------------------------------------
# 1. 공유 Encoder
# ---------------------------------------------------------------------
class Encoder(nn.Module):
    """통합 입력 vocab 사용: x -> enc_output, hidden (입력마다 비-PAD 토큰 필요)"""

    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        hidden_dim=256,
        num_layers=2,
        pad_id=0,
        dropout=0.1,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=pad_id,
        )

        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

    def forward(self, x):
        # x: (batch, Tv)

        # PAD 를 제외한 길이로 packing (배치 내 길이 정렬은 불필요)
        lengths = (x != self.embedding.padding_idx).sum(dim=1)

        embedded = self.embedding(x)

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        packed_output, hidden = self.gru(packed)

        # 가장 긴 실제 문장보다 입력 텐서가 길어도 원래 Tv 를 유지한다.
        enc_output, _ = pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=x.size(1),
        )

        # GRU 출력은 (batch, Tv, 2 * hidden_dim), hidden 은 (2 * num_layers, batch, hidden_dim).
        # 두 방향의 위치별 출력은 이미 원래 입력 순서로 정렬되어 있다.
        # 같은 토큰 위치의 벡터를 합산하므로 역방향 출력을 다시 뒤집으면 안 된다.
        enc_output = enc_output.reshape(
            x.size(0), x.size(1), 2, self.gru.hidden_size
        ).sum(dim=2)
        # hidden 은 각 방향이 문장 전체를 읽은 최종 상태이다.
        # 층별 [정방향, 역방향]을 합산해 단방향 Decoder 에 전달한다.
        # 마지막 토큰의 enc_output 은 역방향의 최종 상태가 아니므로 대신 사용할 수 없다.
        hidden = hidden.reshape(
            self.gru.num_layers, 2, x.size(0), self.gru.hidden_size
        ).sum(dim=1)

        # enc_output: (batch, Tv, hidden_dim)
        # hidden: (num_layers, batch, hidden_dim)
        return enc_output, hidden


# ---------------------------------------------------------------------
# 2. 출력 언어별 Decoder
# ---------------------------------------------------------------------
class Decoder(nn.Module):
    """한 토큰씩 생성하므로 단방향 GRU 사용: (y, hidden) -> logits, hidden"""

    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        hidden_dim=256,
        num_layers=2,
        pad_id=0,
        dropout=0.1,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=pad_id,
        )

        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, y, hidden):
        # y: (batch, Tq)
        # hidden: (num_layers, batch, hidden_dim)

        lengths = (y != self.embedding.padding_idx).sum(dim=1).cpu()
        # 추론 중 PAD 하나가 들어와도 길이 0 으로 인한 packing 오류를 막는다.
        lengths = lengths.clamp(min=1)

        embedded = self.embedding(y)

        packed = pack_padded_sequence(
            embedded,
            lengths,
            batch_first=True,
            enforce_sorted=False,
        )

        # Encoder hidden (또는 직전 decode_step hidden)을 초기 상태로 사용한다.
        packed_out, hidden = self.gru(packed, hidden)

        dec_output, _ = pad_packed_sequence(
            packed_out,
            batch_first=True,
            total_length=y.size(1),
        )

        # logits: (batch, Tq, vocab_size), hidden: (num_layers, batch, hidden_dim)
        logits = self.fc(dec_output)
        return logits, hidden

## 6. seq2seq — 1 enc 3 dec 모델 조립 (담당: 세원)

```
구조
  num ─┐                          ┌─▶ Decoder["num"] ──▶ num
  kor ─┼─▶ 공유 Encoder ──hidden──┼─▶ Decoder["kor"] ──▶ kor
  eng ─┘   (중간 표현)            └─▶ Decoder["eng"] ──▶ eng

- Encoder 하나가 세 언어 입력을 모두 받는다 (입력 vocab 통합)
- 출력 언어는 어떤 Decoder를 쓰느냐로 정해진다
- attention 미사용: Encoder의 마지막 hidden 이 Decoder 로 넘어가는 중간 표현

[가정] model.py (기수)
- encoder(x)         -> enc_output (B, Tv, H), hidden (num_layers, B, H)
- decoder(y, hidden) -> logits (B, Tq, vocab), hidden (num_layers, B, H)   * fc 포함
```

In [ ]:
# ===== src/seq2seq.py =====
PAD_ID = 0
LANGS = ("num", "eng", "kor")
DIRECTIONS = ("num2kor", "kor2num", "num2eng", "eng2num", "kor2eng", "eng2kor")


class Seq2Seq(nn.Module):
    """한 방향 모델: Encoder 1개 + Decoder 1개

    MultiSeq2Seq.direction() 이 만들어 주거나, checkpoint.load_model() 이 돌려준다.
    inference.py (상호) 의 greedy_decode / beam_search / translate 에 그대로 넣을 수 있다.
    """

    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, x, y):
        """학습용 (teacher forcing): x (B, Tv), y (B, Tq) -> logits (B, Tq, vocab)"""
        _, hidden = self.encoder(x)
        logits, _ = self.decoder(y, hidden)
        return logits

    def encode(self, x):
        """추론용: 입력을 한 번만 인코딩 -> enc_output, hidden, x_mask

        enc_output, x_mask 는 attention 이 없어서 decode_step 에서 쓰지 않지만,
        inference.py 의 beam search 가 배치 차원을 복제할 때 쓰므로 그대로 돌려준다.
        """
        enc_output, hidden = self.encoder(x)
        x_mask = x != PAD_ID
        return enc_output, hidden, x_mask

    def decode_step(self, token, hidden, enc_output=None, x_mask=None):
        """한 토큰 디코딩: token (B, 1) -> logits (B, 1, vocab), hidden, None"""
        logits, hidden = self.decoder(token, hidden)
        return logits, hidden, None


class MultiSeq2Seq(nn.Module):
    """공유 Encoder 1개 + 출력 언어별 Decoder 3개 (학습용)"""

    def __init__(self, encoder, decoders):
        super().__init__()
        self.encoder = encoder
        # 일반 dict 가 아니라 ModuleDict 여야 parameters(), to(device), state_dict() 에 포함된다
        self.decoders = nn.ModuleDict(decoders)

    def forward(self, x, y, tgt_lang):
        """학습용: x (B, Tv), y (B, Tq), tgt_lang "num" / "eng" / "kor" -> logits (B, Tq, vocab)"""
        _, hidden = self.encoder(x)
        logits, _ = self.decoders[tgt_lang](y, hidden)
        return logits

    def direction(self, direction):
        """방향 이름 ("kor2eng" 등) -> 그 방향의 Seq2Seq

        복사가 아니라 같은 Encoder / Decoder 객체를 공유한다 (평가·추론용).
        """
        _, tgt_lang = direction.split("2")
        seq = Seq2Seq(self.encoder, self.decoders[tgt_lang])
        # 새로 만든 nn.Module 은 기본이 train 모드라서, 지금 모델의 모드(train / eval)에 맞춘다
        seq.train(self.training)
        return seq


def build_model(src_vocab_size, tgt_vocab_sizes, **model_kwargs):
    """MultiSeq2Seq 생성

    src_vocab_size : 통합 입력 vocab 크기
    tgt_vocab_sizes: {"num": 14, "eng": 36, "kor": 20} 처럼 출력 언어별 vocab 크기
    model_kwargs   : Encoder / Decoder 공통 인자 (embedding_dim, hidden_dim, num_layers, dropout ...)
    """
    encoder = Encoder(src_vocab_size, **model_kwargs)
    decoders = {lang: Decoder(size, **model_kwargs) for lang, size in tgt_vocab_sizes.items()}
    return MultiSeq2Seq(encoder, decoders)

## 7. inference — greedy / beam search · 번역 · 평가 (담당: 상호)

In [ ]:
# ===== src/inference.py =====
# 특수 토큰 id 는 data 셀의 PAD_ID / BOS_ID / EOS_ID (0 / 1 / 2) 를 그대로 쓴다.
# ---------------------------------------------------------------- 내부 유틸
def _eval_mode(fn):
    """eval 모드 + no_grad로 실행하고, 끝나면 원래 모드로 복구"""
    @functools.wraps(fn)
    def wrapper(model, *args, **kwargs):
        was_training = model.training
        model.eval()
        try:
            with torch.no_grad():
                return fn(model, *args, **kwargs)
        finally:
            model.train(was_training)
    return wrapper


def _device(model):
    return next(model.parameters()).device


def _step_logits(logits):
    """(B, 1, V) 또는 (B, V) → (B, V)"""
    return logits.reshape(logits.size(0), -1)


def _select_hidden(hidden, idx):
    """hidden (num_layers, B, H)을 배치 차원(dim 1)에서 idx 순서로 재배열"""
    if isinstance(hidden, tuple):  # LSTM으로 바뀌어도 동작하도록
        return tuple(h.index_select(1, idx) for h in hidden)
    return hidden.index_select(1, idx)


def strip_special(ids):
    """BOS·PAD 제거, EOS에서 자르기"""
    out = []
    for t in ids:
        if t == EOS_ID:
            break
        if t in (BOS_ID, PAD_ID):
            continue
        out.append(t)
    return out


# ---------------------------------------------------------------- greedy
@_eval_mode
def greedy_decode(model, src, max_len=60):
    """
    src: (B, Tv) LongTensor (앞뒤 <bos>, <eos>, 패딩 포함 — 학습 배치와 같은 형식)
    반환: 길이 B 리스트, 각 원소는 토큰 id 리스트 (<bos>/<eos> 제외)
    """
    device = _device(model)
    src = src.to(device)
    B = src.size(0)

    enc_output, hidden, x_mask = model.encode(src)
    token = torch.full((B, 1), BOS_ID, dtype=torch.long, device=device)
    finished = torch.zeros(B, dtype=torch.bool, device=device)
    outputs = []

    for _ in range(max_len):
        logits, hidden, _ = model.decode_step(token, hidden, enc_output, x_mask)
        next_tok = _step_logits(logits).argmax(-1)          # (B,)
        next_tok = next_tok.masked_fill(finished, EOS_ID)      # 끝난 문장은 EOS 유지
        outputs.append(next_tok)
        finished |= next_tok == EOS_ID
        if finished.all():
            break
        token = next_tok.unsqueeze(1)

    seqs = torch.stack(outputs, dim=1).tolist()
    return [strip_special(s) for s in seqs]


# ---------------------------------------------------------------- beam search
@_eval_mode
def beam_search(model, src, beam_width=5, max_len=60, alpha=0.0, return_all=False):
    """
    배치 단위 beam search. 샘플마다 K개 빔을 (B*K) 배치로 펼쳐서 한 번에 디코딩한다.

    src        : (B, Tv) LongTensor
    beam_width : K
    alpha      : 길이 정규화 지수. 최종 점수 = logP / len**alpha (0이면 정규화 없음)
    return_all : True면 샘플마다 [(ids, score), ...] K개를 점수순으로 반환

    반환: 기본은 길이 B 리스트, 각 원소는 최고 점수 빔의 토큰 id 리스트 (<bos>/<eos> 제외)
    """
    device = _device(model)
    src = src.to(device)
    B, K = src.size(0), beam_width

    enc_output, hidden, x_mask = model.encode(src)

    # 샘플마다 K개로 복제: [s0, s0, ..., s1, s1, ...] → (B*K, ...)
    rep = torch.arange(B, device=device).repeat_interleave(K)
    enc_output = enc_output.index_select(0, rep)
    x_mask = x_mask.index_select(0, rep)
    hidden = _select_hidden(hidden, rep)

    # 처음엔 빔 0만 살아 있게 해서 K개 빔이 같은 토큰으로 중복 시작하는 것을 막는다
    scores = torch.full((B, K), float("-inf"), device=device)
    scores[:, 0] = 0.0

    token = torch.full((B * K, 1), BOS_ID, dtype=torch.long, device=device)
    seqs = torch.empty((B * K, 0), dtype=torch.long, device=device)
    finished = torch.zeros(B * K, dtype=torch.bool, device=device)
    lengths = torch.zeros(B * K, device=device)            # EOS 포함 생성 길이
    batch_offset = (torch.arange(B, device=device) * K).unsqueeze(1)  # (B, 1)

    for _ in range(max_len):
        logits, hidden, _ = model.decode_step(token, hidden, enc_output, x_mask)
        logp = F.log_softmax(_step_logits(logits).float(), dim=-1)    # (B*K, V)
        V = logp.size(-1)

        # 끝난 빔은 EOS만 log 1 = 0 → 점수가 그대로 유지되고 계속 EOS만 붙는다
        if finished.any():
            logp[finished] = float("-inf")
            logp[finished, EOS_ID] = 0.0

        cand = (scores.view(-1, 1) + logp).view(B, K * V)
        scores, top_idx = cand.topk(K, dim=-1)               # (B, K)
        beam_idx = top_idx // V                               # 어느 부모 빔에서 왔는지
        tok = top_idx % V                                     # 새로 붙는 토큰

        flat = (batch_offset + beam_idx).view(-1)             # (B*K,) 부모 빔의 펼친 인덱스
        seqs = torch.cat([seqs.index_select(0, flat), tok.view(-1, 1)], dim=1)
        hidden = _select_hidden(hidden, flat)

        prev_finished = finished.index_select(0, flat)
        lengths = lengths.index_select(0, flat) + (~prev_finished).float()
        finished = prev_finished | (tok.view(-1) == EOS_ID) | torch.isinf(scores.view(-1))

        if finished.all():
            break
        token = tok.view(-1, 1)

    final = scores.view(-1)
    if alpha > 0:
        final = final / lengths.clamp(min=1).pow(alpha)
    final = final.view(B, K)
    order = final.argsort(dim=-1, descending=True)

    seqs = seqs.view(B, K, -1).tolist()
    final = final.tolist()
    order = order.tolist()

    if return_all:
        return [[(strip_special(seqs[b][k]), final[b][k]) for k in order[b]] for b in range(B)]
    return [strip_special(seqs[b][order[b][0]]) for b in range(B)]


def decode_batch(model, src, method="beam", beam_width=5, max_len=60, alpha=0.0):
    """method에 따라 greedy / beam 중 하나로 디코딩"""
    if method == "greedy":
        return greedy_decode(model, src, max_len=max_len)
    if method == "beam":
        return beam_search(model, src, beam_width=beam_width, max_len=max_len, alpha=alpha)
    raise ValueError(f"method는 'greedy' 또는 'beam': {method}")


# ---------------------------------------------------------------- 문자열 번역
def _texts_to_tensor(texts, src_vocab, device):
    """문자열 리스트 → 학습 배치와 같은 형식의 (B, Tv) 텐서 (앞뒤 <bos> <eos>, 오른쪽 패딩)"""
    rows = []
    for text in texts:
        ids = list(src_vocab.encode(text))
        # Vocab.encode가 <bos>/<eos>를 붙이는지와 무관하게 동작하도록
        if not ids or ids[0] != BOS_ID:
            ids = [BOS_ID] + ids
        if ids[-1] != EOS_ID:
            ids = ids + [EOS_ID]
        rows.append(ids)
    T = max(len(r) for r in rows)
    padded = [r + [PAD_ID] * (T - len(r)) for r in rows]
    return torch.tensor(padded, dtype=torch.long, device=device)


def translate(model, texts, src_vocab, tgt_vocab, method="beam",
              beam_width=5, max_len=60, alpha=0.0):
    """
    texts: 문자열 하나 또는 문자열 리스트 (숫자도 문자열로! 예: "12345")
    반환: 입력이 문자열이면 문자열, 리스트면 리스트
    """
    single = isinstance(texts, str)
    if single:
        texts = [texts]
    src = _texts_to_tensor(texts, src_vocab, _device(model))
    preds = decode_batch(model, src, method, beam_width, max_len, alpha)
    outs = [tgt_vocab.decode(p) for p in preds]
    return outs[0] if single else outs


# ---------------------------------------------------------------- 평가
def exact_match(hyps, refs):
    """문장 단위 정확 일치율 (앞뒤 공백은 무시)"""
    assert len(hyps) == len(refs)
    if not refs:
        return 0.0
    return sum(h.strip() == r.strip() for h, r in zip(hyps, refs)) / len(refs)


def evaluate(model, loader, tgt_vocab, method="greedy", beam_width=5, max_len=60,
             alpha=0.0, max_batches=None, src_vocab=None, n_errors=5):
    """
    loader가 (src, tgt) 배치를 내보낸다고 가정 (data.py 인터페이스).
    train 중 valid에는 method="greedy"(+max_batches로 일부만), 최종 test에는 "beam" 권장.

    반환: dict(acc, correct, total, errors)
      errors: 틀린 예시 최대 n_errors개 [(src_text, ref, hyp)] — src_vocab을 넘기면 src_text 채움
    """
    correct = total = 0
    errors = []
    for i, (src, tgt) in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        preds = decode_batch(model, src, method, beam_width, max_len, alpha)
        for j, (p, t) in enumerate(zip(preds, tgt.tolist())):
            hyp = tgt_vocab.decode(p)
            ref = tgt_vocab.decode(strip_special(t))
            if hyp.strip() == ref.strip():
                correct += 1
            elif len(errors) < n_errors:
                s = src_vocab.decode(strip_special(src[j].tolist())) if src_vocab else None
                errors.append((s, ref, hyp))
            total += 1
    return {"acc": correct / max(total, 1), "correct": correct, "total": total, "errors": errors}

## 8. checkpoint — 모델 저장 · 로드 (담당: 세원)

In [ ]:
# ===== src/checkpoint.py =====
# ---------------------------------------------------------------------
# Vocab <-> dict
# ---------------------------------------------------------------------
def _vocab_state(vocab, lang=None):
    """Vocab 객체 -> 저장용 dict (itos, stoi, col 같은 기본 자료형만)

    통합 입력 vocab 은 방향마다 입력 언어가 다르므로,
    lang 을 주면 그 방향 파일 안에서는 col(토큰화 방식)을 입력 언어로 바꿔 저장한다.
    """
    state = dict(vars(vocab))
    if lang is not None and "col" in state:
        state["col"] = lang
    return state


def _vocab_from_state(state):
    """저장용 dict -> Vocab 객체 (__init__ 을 거치지 않고 속성만 복원)"""
    vocab = Vocab.__new__(Vocab)
    vocab.__dict__.update(state)
    return vocab


# ---------------------------------------------------------------------
# 1. 제출용 6개 파일
# ---------------------------------------------------------------------
def save_direction(path, model, direction, src_vocab, tgt_vocab, model_kwargs):
    """MultiSeq2Seq 에서 한 방향만 꺼내 저장"""
    src_lang, tgt_lang = direction.split("2")
    torch.save(
        {
            "direction": direction,
            "encoder": model.encoder.state_dict(),
            "decoder": model.decoders[tgt_lang].state_dict(),
            "src_vocab": _vocab_state(src_vocab, src_lang),
            "tgt_vocab": _vocab_state(tgt_vocab),
            "model_config": {
                "src_vocab_size": len(src_vocab),
                "tgt_vocab_size": len(tgt_vocab),
                **model_kwargs,
            },
        },
        path,
    )


def save_all(out_dir, model, src_vocab, tgt_vocabs, model_kwargs, directions=DIRECTIONS):
    """6방향 모두 저장 -> 저장한 경로 리스트

    tgt_vocabs: {"num": Vocab, "eng": Vocab, "kor": Vocab}
    """
    os.makedirs(out_dir, exist_ok=True)
    paths = []
    for direction in directions:
        _, tgt_lang = direction.split("2")
        path = os.path.join(out_dir, f"{direction}.pt")
        save_direction(path, model, direction, src_vocab, tgt_vocabs[tgt_lang], model_kwargs)
        paths.append(path)
    return paths


def load_model(path, device="cpu"):
    """.pt 하나 -> (Seq2Seq, src_vocab, tgt_vocab)

    저장된 config 로 같은 크기의 Encoder / Decoder 를 먼저 만든 뒤 가중치를 넣는다.
    돌려주는 모델은 eval 모드.
    """
    ckpt = torch.load(path, map_location=device)   # GPU 에서 저장한 파일도 CPU 에서 로드 가능

    config = dict(ckpt["model_config"])
    src_vocab_size = config.pop("src_vocab_size")
    tgt_vocab_size = config.pop("tgt_vocab_size")

    encoder = Encoder(src_vocab_size, **config)
    decoder = Decoder(tgt_vocab_size, **config)
    encoder.load_state_dict(ckpt["encoder"])
    decoder.load_state_dict(ckpt["decoder"])

    model = Seq2Seq(encoder, decoder).to(device)
    model.eval()
    return model, _vocab_from_state(ckpt["src_vocab"]), _vocab_from_state(ckpt["tgt_vocab"])


# ---------------------------------------------------------------------
# 2. 학습 중단 대비
# ---------------------------------------------------------------------
def save_training_state(path, model, optimizer, epoch, best_score=None):
    """학습을 이어서 할 수 있게 모델 + optimizer + epoch 저장"""
    torch.save(
        {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "epoch": epoch,
            "best_score": best_score,
        },
        path,
    )


def load_training_state(path, model, optimizer=None, device="cpu"):
    """저장한 학습 상태를 model, optimizer 에 넣고 (epoch, best_score) 반환"""
    state = torch.load(path, map_location=device)
    model.load_state_dict(state["model"])
    if optimizer is not None:
        optimizer.load_state_dict(state["optimizer"])
    return state["epoch"], state["best_score"]

## 9. train — 6방향 혼합 학습 · 검증 (담당: 성현)

In [ ]:
# ===== src/train.py =====
PAD_ID = 0


def batch_loss(target, logits):
    """이미 한 칸 맞춘 정답 (B,T)과 예측 점수 (B,T,V)의 평균 loss."""
    if logits.ndim != 3 or target.ndim != 2 or logits.shape[:2] != target.shape:
        raise ValueError("정답은 (B,T), 예측 점수는 (B,T,V)여야 합니다.")
    return F.cross_entropy(
        logits.reshape(-1, logits.size(-1)), target.reshape(-1), ignore_index=PAD_ID,
    )


def _batch_result(model, x, y, tgt_lang):
    logits = model(x, y, tgt_lang)
    vocab_size = model.decoders[tgt_lang].embedding.num_embeddings
    if logits.ndim != 3 or logits.size(-1) != vocab_size:
        raise ValueError(
            f"{tgt_lang} Decoder 출력 {tuple(logits.shape)}: 마지막 차원은 출력 사전 "
            f"크기 {vocab_size}여야 합니다. model.py의 fc 출력층을 확인하세요."
        )
    target = y[:, 1:]
    loss = batch_loss(target, logits[:, :-1, :])
    if not torch.isfinite(loss):
        raise FloatingPointError("loss가 NaN/Inf입니다. 입력과 비패딩 정답을 확인하세요.")
    return loss, target.ne(PAD_ID).sum().item()


def mixed_batches(loaders, seed):
    """방향 이름만 섞고, 각 loader의 배치는 한 번씩 꺼낸다."""
    order = []
    streams = {}
    for direction in DIRECTIONS:
        loader = loaders[direction]["train"]
        if len(loader) == 0:
            raise ValueError(f"{direction} 학습 배치가 비어 있습니다.")
        order.extend([direction] * len(loader))
        streams[direction] = iter(loader)

    random.Random(seed).shuffle(order)
    for direction in order:
        x, y = next(streams[direction])
        yield direction, x, y


def train_epoch(model, loaders, optimizer, device, seed, grad_clip=1.0):
    """하나의 optimizer로 6방향을 한 바퀴 학습한다."""
    model.train()
    loss_sum, token_count = 0.0, 0
    for direction, x, y in mixed_batches(loaders, seed):
        tgt_lang = direction.split("2")[1]
        optimizer.zero_grad(set_to_none=True)
        loss, count = _batch_result(model, x.to(device), y.to(device), tgt_lang)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip, error_if_nonfinite=True)
        optimizer.step()
        loss_sum += loss.item() * count
        token_count += count
    return loss_sum / token_count


@torch.no_grad()
def validate(model, loaders, device, max_len=60):
    """방향별 valid loss와 greedy로 전체 토큰열이 일치한 비율을 반환한다."""
    model.eval()
    by_direction = {}
    for direction in DIRECTIONS:
        tgt_lang = direction.split("2")[1]
        one_model = model.direction(direction)
        loss_sum, token_count, correct, examples = 0.0, 0, 0, 0
        for x, y in loaders[direction]["valid"]:
            x, y = x.to(device), y.to(device)
            loss, count = _batch_result(model, x, y, tgt_lang)
            loss_sum += loss.item() * count
            token_count += count

            predictions = greedy_decode(one_model, x, max_len=max_len)
            for prediction, gold in zip(predictions, y.tolist()):
                correct += prediction == strip_special(gold)
            examples += y.size(0)
        if examples == 0:
            raise ValueError(f"{direction} 검증 배치가 비어 있습니다.")
        by_direction[direction] = {
            "loss": loss_sum / token_count, "greedy_exact": correct / examples,
            "correct": correct, "examples": examples,
        }
    return by_direction


def _save_epoch(path, model, optimizer, epoch, best_loss):
    # 저장 중 중단돼도 직전까지 완성된 파일을 읽을 수 있게 한다.
    tmp_path = path.with_name(path.name + ".tmp")
    save_training_state(tmp_path, model, optimizer, epoch, best_score=best_loss)
    tmp_path.replace(path)


def train(model, loaders, optimizer, config, device=None, start_epoch=0, ckpt_path=None):
    """노트북 진입점. 매 epoch의 방향별 검증 기록을 리스트로 반환한다.

    start_epoch는 0부터 세며, 재개 시 노트북이 모델과 optimizer를 먼저 복원한다.
    config의 epochs는 목표 총 횟수(기본 30). best 기준은 여섯 방향 valid loss의 평균.
    """
    epochs = config.get("epochs", 30)
    seed = config.get("random_seed", 42)
    grad_clip = config.get("grad_clip", 1.0)
    max_len = config.get("max_len", 60)
    if epochs < 1 or start_epoch < 0 or grad_clip <= 0 or max_len < 1:
        raise ValueError("epochs, grad_clip, max_len은 양수이고 start_epoch는 0 이상이어야 합니다.")
    device = next(model.parameters()).device if device is None else device
    history = []
    best_loss = float("inf")
    best_weights = None

    if ckpt_path is not None:
        ckpt_path = Path(ckpt_path)
        ckpt_path.parent.mkdir(parents=True, exist_ok=True)
        best_path = ckpt_path.with_name(ckpt_path.stem + ".best" + ckpt_path.suffix)
        if start_epoch > 0 and best_path.is_file():
            previous = torch.load(best_path, map_location="cpu", weights_only=True)
            best_loss, best_weights = previous["best_score"], previous["model"]
            del previous

    for epoch in range(start_epoch, epochs):
        train_loss = train_epoch(model, loaders, optimizer, device, seed + epoch, grad_clip)
        valid = validate(model, loaders, device, max_len)
        mean_valid_loss = sum(row["loss"] for row in valid.values()) / len(valid)
        history.append({"epoch": epoch + 1, "train_loss": train_loss,
                        "valid_loss": mean_valid_loss, "valid": valid})

        if mean_valid_loss < best_loss:
            best_loss = mean_valid_loss
            best_weights = {key: value.detach().cpu().clone()
                            for key, value in model.state_dict().items()}
            if ckpt_path is not None:
                _save_epoch(best_path, model, optimizer, epoch, best_loss)
        if ckpt_path is not None:
            _save_epoch(ckpt_path, model, optimizer, epoch, best_loss)

        print(f"epoch {epoch + 1}/{epochs} | train loss {train_loss:.4f} | valid loss {mean_valid_loss:.4f}")
        for direction, row in valid.items():
            print(f"  {direction}: loss {row['loss']:.4f}, greedy exact {row['greedy_exact']:.1%}")

    if best_weights is not None:
        model.load_state_dict(best_weights)
    model.eval()
    return history

## 10. loader 생성

숫자 기준 8:1:1 분할을 6방향이 함께 쓰고, 입력 vocab 은 세 언어를 합친 통합 사전이다.

In [ ]:
loaders, src_vocabs, tgt_vocabs = build_loaders(CONFIG, DIRECTIONS, limit=LIMIT)
src_vocab = src_vocabs["num"]   # 세 입력 vocab 은 id 가 같다. 모델 크기·저장용 대표

print("입력 vocab:", len(src_vocab), "| 출력 vocab:", {l: len(v) for l, v in tgt_vocabs.items()})
for d in DIRECTIONS:
    print(f"{d:8s}", {k: len(loaders[d][k].dataset) for k in ("train", "valid", "test")})

## 11. 모델: 공유 Encoder 1개 + Decoder 3개

In [ ]:
model = build_model(len(src_vocab), {l: len(v) for l, v in tgt_vocabs.items()}, **MODEL_KWARGS).to(device)
print(model)
print("파라미터:", f"{sum(p.numel() for p in model.parameters()):,}")

## 12. 학습

In [ ]:
# 셀을 실행할 때마다 마지막으로 저장된 지점부터 EPOCHS_PER_RUN 만큼 더 학습한다.
# train() 이 epoch 마다 last.pt(마지막 상태)와 last.best.pt(valid loss 최저)를 CKPT_DIR(Colab 은 Drive)에 저장한다.
# 세션이 끊기면 1번 셀부터 다시 실행한다 -> 저장된 지점에서 이어서 학습한다.
# 처음부터 다시 학습하려면 2번 셀의 FRESH = True 로 실행한다 -> last.pt, last.best.pt 를 지운다 (모델 크기·vocab 을 바꿨을 때도 반드시 지울 것).
# 학습이 끝나면 model 에는 지금까지 valid loss 가 가장 낮았던 가중치가 들어 있다 (13·14번 셀은 이 모델로 평가·저장).
EPOCHS_PER_RUN = 5                                                # 이 셀 한 번 실행에 돌릴 epoch 수
LAST = os.path.join(CKPT_DIR, "last.pt")
BEST = os.path.join(CKPT_DIR, "last.best.pt")
if FRESH:
    for path in (LAST, BEST):
        if os.path.exists(path):
            os.remove(path)

optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])
start_epoch = 0
if os.path.exists(LAST):                                          # 저장된 지점이 있으면 모델·optimizer 를 거기로 되돌림
    last_epoch, best = load_training_state(LAST, model, optimizer, device)
    start_epoch = last_epoch + 1
    best_txt = f"{best:.4f}" if best is not None else "-"
    print(f"[이어서 학습] {LAST} 에서 불러옴 -> {start_epoch + 1}번째 epoch부터 (완료 {start_epoch} epoch, best valid loss {best_txt})")
else:
    print(f"[처음부터 학습] {LAST} 없음 -> 1번째 epoch부터")
end_epoch = start_epoch + EPOCHS_PER_RUN
print(f"이번 실행: {start_epoch + 1} ~ {end_epoch}번째 epoch ({EPOCHS_PER_RUN} epoch)")

run_config = {**CONFIG, "epochs": end_epoch}                      # train() 은 config["epochs"] 를 목표 총 epoch 수로 쓴다
history = train(model, loaders, optimizer, run_config, device=device,
                start_epoch=start_epoch, ckpt_path=LAST)
print(f"학습 완료 -> 마지막 상태: {LAST} / best: {BEST}")

## 13. 방향별 test 평가 (beam search)

In [ ]:
model.eval()
results = {}
for d in DIRECTIONS:
    s, t = d.split("2")
    r = evaluate(model.direction(d), loaders[d]["test"], tgt_vocabs[t],
                 method="beam", beam_width=5, src_vocab=src_vocabs[s])
    results[d] = r["acc"]
    print(f"{d:8s} acc {r['acc']:.4f} ({r['correct']}/{r['total']})  틀린 예: {r['errors'][:2]}")

## 14. 제출용 `.pt` 6개 저장

Colab 에서는 Drive `MyDrive/2026-nlp/checkpoints/final/` 에 저장된다. 여기서 내려받아 `project01/` 에 올린다.

In [ ]:
paths = save_all(SUBMIT_DIR, model, src_vocab, tgt_vocabs, MODEL_KWARGS)
for p in paths:
    print(p, f"{os.path.getsize(p) / 1e6:.1f} MB")

## 15. 저장한 파일만으로 번역 확인 + 직접 입력

In [ ]:
# 메모리의 model 이 아니라 SUBMIT_DIR 의 .pt 파일만 불러와 번역한다.
# (학습을 다시 하지 않아도 1 ~ 9번 셀만 실행한 뒤 이 셀을 실행하면 동작)
# ① 6방향 자동 점검: 12708 을 세 언어로 넣어 6개 파일이 모두 로드·번역되는지 확인
# ② 직접 입력: "방향 문장" 형식 (예: num2kor 12708 / kor2eng 만 이천칠백팔), 빈 줄을 입력하면 종료
INTERACTIVE = True                                                # 직접 입력을 건너뛰려면 False

_cache = {}

def tr(direction, text, method="beam"):
    """저장된 .pt 로 번역. direction: num2kor 등 / text: 문자열 또는 문자열 리스트 (숫자도 문자열)"""
    if direction not in _cache:                                   # 방향별 파일은 처음 한 번만 로드
        _cache[direction] = load_model(os.path.join(SUBMIT_DIR, f"{direction}.pt"), device=device)
    m, sv, tv = _cache[direction]
    return translate(m, text, sv, tv, method=method)

# ① 6방향 자동 점검
examples = {"num": "12708", "kor": "만 이천칠백팔", "eng": "twelve thousand seven hundred eight"}
for d in DIRECTIONS:
    src, _ = d.split("2")
    print(f"{d:8s} {examples[src]!r} -> {tr(d, examples[src])!r}")

# ② 직접 입력
if INTERACTIVE:
    print("\n직접 입력: '방향 문장' (예: num2kor 189), 빈 줄을 입력하면 종료")
    while True:
        line = input("방향 문장> ").strip()
        if not line:
            break
        direction, _, text = line.partition(" ")
        try:
            print(" ->", tr(direction, text))
        except FileNotFoundError:
            print(" 파일이 없습니다. 방향 이름(num2kor, kor2num, num2eng, eng2num, kor2eng, eng2kor)과"
                  " 14번 셀로 final/ 에 저장했는지 확인하세요.")